# 01 — Data Understanding

**Football Odds Intelligence & Match Analytics**

---

## Purpose

This notebook performs the initial data understanding phase:

1. Load all raw datasets
2. Profile each dataset (shape, types, missing values, duplicates)
3. Validate referential integrity between tables
4. Identify unique values for key categorical columns
5. Generate summary statistics
6. Document findings for downstream phases

**Rule:** We observe and document — we do NOT modify data here.

In [ ]:
# Standard imports
import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import src/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import (
    load_matches, load_match_closing, 
    load_odds_1x2, load_odds_ah, load_odds_ou
)

# Display settings
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print(f"Python: {sys.version}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"Project root: {project_root}")

---
## 1. Load Datasets

We load the two smaller tables fully, and only the metadata/shape of the
large odds time-series files to avoid memory issues.

In [ ]:
# Load the two manageable datasets fully
matches = load_matches()
closing = load_match_closing()

print(f"matches:       {matches.shape[0]:>12,} rows × {matches.shape[1]} cols")
print(f"match_closing: {closing.shape[0]:>12,} rows × {closing.shape[1]} cols")

In [ ]:
# For the large odds files, load a small sample to inspect schema
# We use match_ids from the first 5 matches for inspection
sample_ids = matches['match_id'].head(5).tolist()

odds_1x2_sample = load_odds_1x2(match_ids=sample_ids)
odds_ah_sample = load_odds_ah(match_ids=sample_ids)
odds_ou_sample = load_odds_ou(match_ids=sample_ids)

print(f"\nodds_1x2 sample: {odds_1x2_sample.shape[0]:>8,} rows × {odds_1x2_sample.shape[1]} cols  (for {len(sample_ids)} matches)")
print(f"odds_ah sample:  {odds_ah_sample.shape[0]:>8,} rows × {odds_ah_sample.shape[1]} cols  (for {len(sample_ids)} matches)")
print(f"odds_ou sample:  {odds_ou_sample.shape[0]:>8,} rows × {odds_ou_sample.shape[1]} cols  (for {len(sample_ids)} matches)")

---
## 2. Dataset Profiling

For each dataset we examine:
- Column names and data types
- Missing values (count and percentage)
- Duplicate rows
- Sample data

### 2.1 matches.csv — Match Metadata & Results

In [ ]:
def profile_dataset(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Generate a profiling summary for a DataFrame.
    
    Returns a DataFrame with one row per column showing:
    dtype, non-null count, null count, null %, unique values, and a sample value.
    """
    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.notna().sum(),
        'null_count': df.isna().sum(),
        'null_pct': (df.isna().sum() / len(df) * 100).round(2),
        'n_unique': df.nunique(),
        'sample_value': df.iloc[0] if len(df) > 0 else None
    })
    
    total_rows = len(df)
    total_dupes = df.duplicated().sum()
    
    print(f"{'='*60}")
    print(f" PROFILE: {name}")
    print(f"{'='*60}")
    print(f" Rows:       {total_rows:>12,}")
    print(f" Columns:    {df.shape[1]:>12}")
    print(f" Duplicates: {total_dupes:>12,}  ({total_dupes/total_rows*100:.2f}%)")
    print(f" Memory:     {df.memory_usage(deep=True).sum() / 1024**2:>12.2f} MB")
    print(f"{'='*60}")
    
    return profile


matches_profile = profile_dataset(matches, 'matches')
matches_profile

In [ ]:
# First 5 rows of matches
matches.head()

In [ ]:
# Check match_id uniqueness (should be the primary key)
match_id_unique = matches['match_id'].is_unique
print(f"match_id is unique: {match_id_unique}")
if not match_id_unique:
    dupes = matches[matches['match_id'].duplicated(keep=False)]
    print(f"Duplicate match_ids: {dupes['match_id'].nunique()}")
    print(dupes.head(10))

In [ ]:
# Summary statistics for numerical columns
matches.describe()

In [ ]:
# Categorical column value counts
print("=" * 50)
print("UNIQUE LEAGUES:", matches['league_name'].nunique())
print("=" * 50)
print(matches['league_name'].value_counts().head(20))

print("\n" + "=" * 50)
print("UNIQUE SEASONS:", matches['season'].nunique())
print("=" * 50)
print(matches['season'].value_counts().sort_index())

print("\n" + "=" * 50)
print("LABEL BOOKMAKERS:")
print("=" * 50)
print(matches['label_book'].value_counts())

print("\n" + "=" * 50)
print("NEUTRAL VENUE:")
print("=" * 50)
print(matches['neutral'].value_counts())

In [ ]:
# Date range
print(f"Earliest match: {matches['kickoff_utc'].min()}")
print(f"Latest match:   {matches['kickoff_utc'].max()}")
print(f"Date range:     {(matches['kickoff_utc'].max() - matches['kickoff_utc'].min()).days} days")

In [ ]:
# Derive match outcome for initial exploration
# H = Home Win, D = Draw, A = Away Win
def derive_match_outcome(row):
    """Derive match outcome from full-time scores."""
    if pd.isna(row['ft_home']) or pd.isna(row['ft_away']):
        return np.nan
    if row['ft_home'] > row['ft_away']:
        return 'H'
    elif row['ft_home'] == row['ft_away']:
        return 'D'
    else:
        return 'A'

matches['outcome'] = matches.apply(derive_match_outcome, axis=1)
print("Match Outcome Distribution:")
print(matches['outcome'].value_counts())
print(f"\nMissing outcomes: {matches['outcome'].isna().sum()}")
print(f"\nOutcome proportions:")
print(matches['outcome'].value_counts(normalize=True).round(4))

In [ ]:
# Logical consistency check: half-time goals <= full-time goals
ht_ft_valid_home = (matches['ht_home'] <= matches['ft_home']).all()
ht_ft_valid_away = (matches['ht_away'] <= matches['ft_away']).all()
print(f"HT ≤ FT (home): {ht_ft_valid_home}")
print(f"HT ≤ FT (away): {ht_ft_valid_away}")

if not ht_ft_valid_home or not ht_ft_valid_away:
    invalid = matches[
        (matches['ht_home'] > matches['ft_home']) | 
        (matches['ht_away'] > matches['ft_away'])
    ]
    print(f"\nInvalid rows: {len(invalid)}")
    print(invalid[['match_id', 'home_team', 'away_team', 'ht_home', 'ht_away', 'ft_home', 'ft_away']].head(10))

### 2.2 match_closing.csv — Opening & Closing Odds per Bookmaker

In [ ]:
closing_profile = profile_dataset(closing, 'match_closing')
closing_profile

In [ ]:
closing.head()

In [ ]:
# Bookmaker distribution
print("BOOKMAKERS in match_closing:")
print(closing['bookmaker'].value_counts())

# Average rows per match
books_per_match = closing.groupby('match_id')['bookmaker'].nunique()
print(f"\nBookmakers per match:")
print(f"  Mean: {books_per_match.mean():.2f}")
print(f"  Min:  {books_per_match.min()}")
print(f"  Max:  {books_per_match.max()}")
print(f"\nDistribution:")
print(books_per_match.value_counts().sort_index())

In [ ]:
# Check for duplicate (match_id, bookmaker) pairs
dupes_closing = closing.duplicated(subset=['match_id', 'bookmaker']).sum()
print(f"Duplicate (match_id, bookmaker) pairs: {dupes_closing}")

In [ ]:
# Check odds validity: all decimal odds should be > 1.0
odds_cols_1x2 = ['open_1x2_home', 'open_1x2_draw', 'open_1x2_away',
                 'close_1x2_home', 'close_1x2_draw', 'close_1x2_away']

for col in odds_cols_1x2:
    non_null = closing[col].dropna()
    invalid = (non_null <= 1.0).sum()
    print(f"{col:25s}  non-null: {len(non_null):>8,}  invalid (≤1.0): {invalid:>6,}  min: {non_null.min():.4f}  max: {non_null.max():.4f}")

In [ ]:
# Summary statistics for odds columns
closing[odds_cols_1x2].describe()

In [ ]:
# Missing value analysis by bookmaker
# (Pinnacle may have missing AH/OU data)
print("Missing values by bookmaker (AH and OU columns):")
ah_ou_cols = ['open_ah_line', 'open_ah_home', 'open_ah_away',
              'close_ah_line', 'close_ah_home', 'close_ah_away',
              'open_ou_line', 'open_ou_over', 'open_ou_under',
              'close_ou_line', 'close_ou_over', 'close_ou_under']

missing_by_book = closing.groupby('bookmaker')[ah_ou_cols].apply(
    lambda x: x.isna().sum()
)
print(missing_by_book)

### 2.3 odds_1x2 — 1X2 Odds Time Series (Sample)

In [ ]:
odds_1x2_profile = profile_dataset(odds_1x2_sample, 'odds_1x2 (sample)')
odds_1x2_profile

In [ ]:
odds_1x2_sample.head(10)

In [ ]:
# Phase values in odds_1x2
print("Phase values:")
print(odds_1x2_sample['phase'].value_counts())

# Average observations per match per bookmaker
obs_per_match_book = odds_1x2_sample.groupby(['match_id', 'bookmaker']).size()
print(f"\nOdds observations per (match, bookmaker):")
print(f"  Mean: {obs_per_match_book.mean():.1f}")
print(f"  Min:  {obs_per_match_book.min()}")
print(f"  Max:  {obs_per_match_book.max()}")

### 2.4 odds_ah — Asian Handicap Odds Time Series (Sample)

In [ ]:
odds_ah_profile = profile_dataset(odds_ah_sample, 'odds_ah (sample)')
odds_ah_profile

In [ ]:
odds_ah_sample.head(10)

In [ ]:
# Suspended flag distribution
print("Suspended flag (odds_ah):")
print(odds_ah_sample['suspended'].value_counts())

print("\nPhase values:")
print(odds_ah_sample['phase'].value_counts())

### 2.5 odds_ou — Over/Under Odds Time Series (Sample)

In [ ]:
odds_ou_profile = profile_dataset(odds_ou_sample, 'odds_ou (sample)')
odds_ou_profile

In [ ]:
odds_ou_sample.head(10)

---
## 3. Referential Integrity Checks

Verify that all `match_id` values in child tables exist in the `matches` table.

In [ ]:
match_ids_master = set(matches['match_id'])

# Check match_closing
closing_ids = set(closing['match_id'])
closing_orphans = closing_ids - match_ids_master
closing_missing = match_ids_master - closing_ids

print(f"match_closing → matches referential integrity:")
print(f"  match_ids in closing but NOT in matches (orphans): {len(closing_orphans)}")
print(f"  match_ids in matches but NOT in closing (missing): {len(closing_missing)}")

# Check odds samples
for name, df_sample in [('odds_1x2', odds_1x2_sample), 
                         ('odds_ah', odds_ah_sample),
                         ('odds_ou', odds_ou_sample)]:
    sample_ids_check = set(df_sample['match_id'])
    orphans = sample_ids_check - match_ids_master
    print(f"\n{name} sample → matches: orphan match_ids = {len(orphans)}")

---
## 4. Missing Value Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Matches missing values
matches_missing = matches.isna().sum()
matches_missing = matches_missing[matches_missing > 0].sort_values(ascending=True)
if len(matches_missing) > 0:
    matches_missing.plot.barh(ax=axes[0], color='#e74c3c')
    axes[0].set_title('Missing Values — matches', fontsize=14)
    axes[0].set_xlabel('Count')
    for i, v in enumerate(matches_missing):
        axes[0].text(v + 50, i, f'{v:,} ({v/len(matches)*100:.1f}%)', va='center', fontsize=9)
else:
    axes[0].text(0.5, 0.5, 'No missing values', transform=axes[0].transAxes,
                ha='center', va='center', fontsize=14)
    axes[0].set_title('Missing Values — matches', fontsize=14)

# Match closing missing values
closing_missing_vals = closing.isna().sum()
closing_missing_vals = closing_missing_vals[closing_missing_vals > 0].sort_values(ascending=True)
if len(closing_missing_vals) > 0:
    closing_missing_vals.plot.barh(ax=axes[1], color='#3498db')
    axes[1].set_title('Missing Values — match_closing', fontsize=14)
    axes[1].set_xlabel('Count')
    for i, v in enumerate(closing_missing_vals):
        axes[1].text(v + 50, i, f'{v:,} ({v/len(closing)*100:.1f}%)', va='center', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'No missing values', transform=axes[1].transAxes,
                ha='center', va='center', fontsize=14)
    axes[1].set_title('Missing Values — match_closing', fontsize=14)

plt.tight_layout()
plt.savefig(project_root / 'reports' / 'figures' / 'missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/missing_values.png")

---
## 5. Initial Distributions

In [ ]:
# Match outcome distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Outcome bar chart
outcome_counts = matches['outcome'].value_counts()
colors = {'H': '#2ecc71', 'D': '#f39c12', 'A': '#e74c3c'}
outcome_counts.plot.bar(ax=axes[0], color=[colors.get(x, '#95a5a6') for x in outcome_counts.index])
axes[0].set_title('Match Outcomes', fontsize=14)
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Home Win', 'Draw', 'Away Win'], rotation=0)
for i, v in enumerate(outcome_counts):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(matches)*100:.1f}%)', ha='center', fontsize=10)

# Total goals distribution
total_goals = matches['ft_home'] + matches['ft_away']
total_goals.value_counts().sort_index().plot.bar(ax=axes[1], color='#3498db')
axes[1].set_title('Total Goals per Match', fontsize=14)
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Goals')

# Matches per season
season_counts = matches['season'].value_counts().sort_index()
season_counts.plot.bar(ax=axes[2], color='#9b59b6')
axes[2].set_title('Matches per Season', fontsize=14)
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(project_root / 'reports' / 'figures' / 'initial_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/initial_distributions.png")

In [ ]:
# Closing odds distribution (1X2)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (col, label) in enumerate([
    ('close_1x2_home', 'Home Win Odds'),
    ('close_1x2_draw', 'Draw Odds'),
    ('close_1x2_away', 'Away Win Odds')
]):
    data = closing[col].dropna()
    # Clip extreme outliers for visualization
    data_clipped = data[data <= data.quantile(0.99)]
    axes[i].hist(data_clipped, bins=80, color='#2c3e50', alpha=0.7, edgecolor='white')
    axes[i].set_title(f'Closing {label}', fontsize=13)
    axes[i].set_xlabel('Decimal Odds')
    axes[i].set_ylabel('Count')
    axes[i].axvline(data.median(), color='red', linestyle='--', alpha=0.7, label=f'Median: {data.median():.2f}')
    axes[i].legend()

plt.tight_layout()
plt.savefig(project_root / 'reports' / 'figures' / 'closing_odds_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/closing_odds_distribution.png")

---
## 6. Bookmaker Overround (Margin) Analysis

The overround is the bookmaker's built-in margin:

$$\text{Overround} = \frac{1}{\text{home}} + \frac{1}{\text{draw}} + \frac{1}{\text{away}} - 1$$

A fair market has overround = 0. Bookmakers typically add 2–10%.

In [ ]:
# Calculate overround for closing 1X2 odds
closing_valid = closing.dropna(subset=['close_1x2_home', 'close_1x2_draw', 'close_1x2_away'])

closing_valid = closing_valid.copy()
closing_valid['overround'] = (
    1 / closing_valid['close_1x2_home'] + 
    1 / closing_valid['close_1x2_draw'] + 
    1 / closing_valid['close_1x2_away'] - 1
) * 100  # Express as percentage

print("Closing 1X2 Overround (%) by Bookmaker:")
print(closing_valid.groupby('bookmaker')['overround'].describe().round(2))

In [ ]:
# Overround distribution by bookmaker
fig, ax = plt.subplots(figsize=(12, 6))
bookmakers = closing_valid['bookmaker'].unique()

for book in sorted(bookmakers):
    data = closing_valid[closing_valid['bookmaker'] == book]['overround']
    # Clip for visualization
    data = data[(data > -5) & (data < 30)]
    ax.hist(data, bins=60, alpha=0.5, label=f'{book} (median: {data.median():.1f}%)')

ax.set_title('Bookmaker Overround Distribution (Closing 1X2)', fontsize=14)
ax.set_xlabel('Overround (%)')
ax.set_ylabel('Count')
ax.axvline(0, color='black', linestyle='--', alpha=0.5, label='Fair market (0%)')
ax.legend()
plt.tight_layout()
plt.savefig(project_root / 'reports' / 'figures' / 'overround_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/overround_distribution.png")

---
## 7. Odds Movement Sample Visualisation

Let's visualise how 1X2 odds evolve for a single match across bookmakers.

In [ ]:
# Pick the first match and plot its odds movement
first_match_id = sample_ids[0]
first_match_info = matches[matches['match_id'] == first_match_id].iloc[0]

print(f"Match: {first_match_info['home_team']} vs {first_match_info['away_team']}")
print(f"League: {first_match_info['league_name']}")
print(f"Kickoff: {first_match_info['kickoff_utc']}")
print(f"Score: {first_match_info['ft_home']}-{first_match_info['ft_away']}")

match_odds = odds_1x2_sample[odds_1x2_sample['match_id'] == first_match_id].copy()
match_odds['ts_utc'] = pd.to_datetime(match_odds['ts_utc'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for book in match_odds['bookmaker'].unique():
    book_data = match_odds[match_odds['bookmaker'] == book].sort_values('ts_utc')
    axes[0].plot(book_data['ts_utc'], book_data['home'], marker='.', markersize=3, label=book, alpha=0.8)
    axes[1].plot(book_data['ts_utc'], book_data['draw'], marker='.', markersize=3, label=book, alpha=0.8)
    axes[2].plot(book_data['ts_utc'], book_data['away'], marker='.', markersize=3, label=book, alpha=0.8)

titles = ['Home Win Odds', 'Draw Odds', 'Away Win Odds']
for i, ax in enumerate(axes):
    ax.set_title(titles[i], fontsize=13)
    ax.set_xlabel('Time')
    ax.set_ylabel('Decimal Odds')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle(
    f"{first_match_info['home_team']} vs {first_match_info['away_team']} — "
    f"Odds Movement (Score: {int(first_match_info['ft_home'])}-{int(first_match_info['ft_away'])})",
    fontsize=15, y=1.02
)
plt.tight_layout()
plt.savefig(project_root / 'reports' / 'figures' / 'sample_odds_movement.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/sample_odds_movement.png")

---
## 8. Full Dataset Row Counts (Large Files)

We already know from Phase 0 discovery:
- `odds_1x2`: 11,607,578 rows
- `odds_ah`:  15,614,752 rows  
- `odds_ou`:   9,807,873 rows

Let's verify by reading just the metadata from the Parquet files.

In [ ]:
import pyarrow.parquet as pq

raw_dir = project_root / 'data' / 'raw'

for name in ['odds_1x2', 'odds_ah', 'odds_ou']:
    pf = pq.ParquetFile(raw_dir / f'{name}.parquet')
    meta = pf.metadata
    schema = pf.schema_arrow
    print(f"\n{'='*50}")
    print(f" {name}.parquet")
    print(f"{'='*50}")
    print(f" Rows:         {meta.num_rows:>12,}")
    print(f" Row Groups:   {meta.num_row_groups:>12}")
    print(f" Columns:      {meta.num_columns:>12}")
    print(f" Schema:")
    for i in range(len(schema)):
        print(f"   {schema.field(i)}")

---
## 9. Key Findings Summary

### Dataset Summary

| Aspect | Finding |
|--------|--------|
| Total matches | 63,597 |
| Total odds observations | ~37 million |
| Bookmakers | 4 (Bet365, Pinnacle, SBOBET, Singbet) |
| Market types | 3 (1X2, Asian Handicap, Over/Under) |
| Match outcomes available | ✅ (ft_home, ft_away → H/D/A) |
| Opening & closing odds | ✅ (match_closing table) |
| Time-series odds movements | ✅ (odds_1x2, odds_ah, odds_ou) |

### Data Quality Observations

Run this notebook to populate findings above.

### Next Steps

1. **Phase 3:** Build data cleaning pipeline based on issues found
2. Address missing values (especially Pinnacle AH/OU gaps)
3. Derive match outcome column permanently
4. Standardise timestamps
5. Create cleaned datasets in data/processed/

In [ ]:
print("\n" + "="*60)
print(" PHASE 2 — DATA UNDERSTANDING COMPLETE")
print("="*60)
print("\nFiles generated:")
print("  reports/figures/missing_values.png")
print("  reports/figures/initial_distributions.png")
print("  reports/figures/closing_odds_distribution.png")
print("  reports/figures/overround_distribution.png")
print("  reports/figures/sample_odds_movement.png")
print("\nNext: Run 02_data_cleaning.ipynb")